# Klucaro

**Setup:** Settings → Accelerator → GPU T4 x2

**Files cần upload:** `config.py`, `model_fcn.py`, `caro_agent_az.py`, `caro_env.py`, `greedy_agent.py`

---
## Setup: Clone repo + kiểm tra

In [1]:
import tensorflow as tf
import os, sys, shutil, glob
from config import BOARD_SIZE, BUFFER_SIZE, TRAIN_EVERY, TRAIN_BATCH, TRAIN_EPOCHS
from caro_env import CaroEnv
from caro_agent_az import CaroAgent
from greedy_agent import GreedyAgent
import time
import numpy as np

In [1]:
import os, sys, shutil, glob

# ═══════════════════════════════════════════════════════════
# SỬA 2 DÒNG NÀY CHO ĐÚNG REPO CỦA BẠN
# ═══════════════════════════════════════════════════════════
GITHUB_URL = 'https://github.com/unkluco/klucaro.git'   # ← URL repo
REPO_NAME  = 'klucaro'                                    # ← tên thư mục sau clone
# ═══════════════════════════════════════════════════════════

WORK_DIR = '/kaggle/working'
CKPT_DIR = os.path.join(WORK_DIR, 'checkpoints')
REPO_DIR = os.path.join(WORK_DIR, REPO_NAME)

# ── 1. Clone repo ──
print('=== SETUP ===')
if os.path.exists(REPO_DIR):
    print(f'Repo đã tồn tại, pull mới nhất...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print(f'Clone {GITHUB_URL}...')
    ret = os.system(f'git clone {GITHUB_URL} {REPO_DIR}')
    if ret != 0:
        print('\n✗ Clone thất bại!')
        print('  Kiểm tra: repo public? URL đúng?')
        print('  Nếu repo private: dùng token')
        print('  GITHUB_URL = "https://TOKEN@github.com/username/repo.git"')

# Liệt kê files
if os.path.exists(REPO_DIR):
    print(f'\nFiles trong {REPO_DIR}:')
    for f in sorted(os.listdir(REPO_DIR)):
        if not f.startswith('.'):
            print(f'  {f}')

# ── 2. Copy .py sang working ──
py_files = glob.glob(os.path.join(REPO_DIR, '*.py'))
for src in py_files:
    shutil.copy2(src, WORK_DIR)
print(f'\n✓ Copied {len(py_files)} .py files → {WORK_DIR}')

# Copy .keras nếu có (để tiếp tục train)
keras_files = glob.glob(os.path.join(REPO_DIR, '**', '*.keras'), recursive=True)
for src in keras_files:
    shutil.copy2(src, WORK_DIR)
if keras_files:
    print(f'✓ Copied {len(keras_files)} .keras checkpoints')

# ── 3. Thư mục + path ──
os.makedirs(CKPT_DIR, exist_ok=True)
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# ── 4. Check GPU ──
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\nGPUs: {len(gpus)}')
for g in gpus: print(f'  {g}')

# ── 5. Test import ──
try:
    from config import BOARD_SIZE, BUFFER_SIZE, TRAIN_EVERY, TRAIN_BATCH, TRAIN_EPOCHS
    from caro_env import CaroEnv
    from caro_agent_az import CaroAgent
    from greedy_agent import GreedyAgent
    print(f'\n✓ Import OK')
except ImportError as e:
    print(f'\n✗ Import lỗi: {e}')

# ── 6. Quick test ──
try:
    import time
    import numpy as np
    _t = CaroAgent(board_size=9, base_filters=16, n_res=1, n_sim=2, n_parallel=1, buffer_size=100)
    r, c = _t.get_action(np.zeros((9,9), dtype=np.float32), player=1)
    print(f'  Agent test: chọn ({r},{c}) ✓')
    del _t
except Exception as e:
    print(f'  Agent test lỗi: {e}')

print('\n=== SETUP XONG ===')

=== SETUP ===
Repo đã tồn tại, pull mới nhất...

Files trong /kaggle/working\klucaro:
  README.md
  caro_agent_az.py
  caro_env.py
  config.py
  greedy_agent.py
  main.ipynb
  model_fcn.py

✓ Copied 5 .py files → /kaggle/working

GPUs: 0

✓ Import OK
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Strategy: 1 replica(s)

  Agent test: chọn (3,4) ✓

=== SETUP XONG ===


---
## Bước 1: Pretrain từ Greedy


In [2]:
# Tạo strategy để train đa GPU (nếu có)
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


In [3]:
agent_9  = CaroAgent(
    board_size=9, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)
greedy_9 = GreedyAgent(board_size=9)
env_9    = CaroEnv(board_size=9)

print(f'Params: {agent_9.model.count_params():,}')

Strategy: 1 replica(s)
Params: 1,780,176


In [ ]:
t0 = time.time()
agent_9.pretrain_from_greedy(
    greedy_9,
    n_games        = 7000,
    batch_size     = 64,
    temperature    = 1.2,
    random_opening = 2,
    skip_first     = 2,
)
print(f'Pretrain: {time.time()-t0:.0f}s')

Thu thập positions từ 1 ván greedy vs greedy...
  → 78 positions
Train 78 positions, batch=64


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\optimizers\base_optimizer.py:870: UserWarning: Gradients do not exist for variables ['conv2d_21/kernel', 'batch_normalization_19/gamma', 'batch_normalization_19/beta', 'dense_2/kernel', 'dense_2/bias', 'dense_3/kernel', 'dense_3/bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


  [78/78]  avg loss = 4.6722
✓ Pretrain xong — 1 ván → 78 positions — avg loss = 4.6722
Pretrain: 3s


In [ ]:
env_9.play_n(agent_9, greedy_9, n=20)
agent_9.save(os.path.join(CKPT_DIR, 'pretrain_9.keras'))

KeyboardInterrupt: 

---
## Bước 1.5: Pretrain 9x9 -> Pretrain 15x15

In [ ]:
agent_15  = CaroAgent(
    board_size=15, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)
greedy_15 = GreedyAgent(board_size=15)
env_15    = CaroEnv(board_size=15)

agent_15.copy_weights_from(agent_9)

print(f'Params: {agent_15.model.count_params():,}')

In [ ]:
t0 = time.time()
agent_15.pretrain_from_greedy(
    greedy_15,
    n_games        = 1000,
    batch_size     = 64,
    temperature    = 1.2,
    random_opening = 2,
    skip_first     = 2,
)
print(f'Pretrain: {time.time()-t0:.0f}s')

In [ ]:
env_15.play_n(agent_15, greedy_15, n=20)
agent_15.save(os.path.join(CKPT_DIR, 'pretrain_15.keras'))

---
## Bước 2: Train MCTS 15×15 với Replay Buffer

In [ ]:
agent_15_2 = CaroAgent(
    board_size=15, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)
agent_15_2.copy_weights_from(agent_15)

t0 = time.time()
env_15.train_with_buffer(
    agent_15, agent_15_2,
    n           = 2000,
    train_every = 20,
    batch_size  = 256,
    epochs      = 3,
    ckpt_every  = 250,
    plot_every  = 100,
)
print(f'Train 15x15: {time.time()-t0:.0f}s')

In [ ]:
env_15.play_n(agent_15, greedy_15, n=20)
agent_15.save(os.path.join(CKPT_DIR, 'train_15.keras'))

---
## Bước 3: Curriculum → 25×25

In [ ]:
agent_25  = CaroAgent(
    board_size=25, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)

agent_25_2 = CaroAgent(
    board_size=25, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)

greedy_25 = GreedyAgent(board_size=25)
env_25    = CaroEnv(board_size=25)

agent_25.copy_weights_from(agent_15)
agent_25_2.copy_weights_from(agent_15)

In [ ]:
t0 = time.time()
env_25.train_with_buffer(
    agent_25, agent_25_2,
    n           = 1000,
    train_every = 20,
    batch_size  = 256,
    epochs      = 3,
    ckpt_every  = 250,
    plot_every  = 50,
)
print(f'Train 25x25: {time.time()-t0:.0f}s')

In [ ]:
agent_25.save(os.path.join(CKPT_DIR, 'train_25.keras'))
env_25.play_n(agent_25, greedy_25, n=20)

---
## Bước 4: Curriculum → 40×40

In [ ]:
agent_40  = CaroAgent(
    board_size=40, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)

agent_40_2 = CaroAgent(
    board_size=40, base_filters=128, n_res=6,
    n_sim=40, n_parallel=8, buffer_size=BUFFER_SIZE, strategy=strategy
)

greedy_40 = GreedyAgent(board_size=40)
env_40    = CaroEnv(board_size=40)

agent_40.copy_weights_from(agent_25)
agent_40_2.copy_weights_from(agent_25)

In [ ]:
t0 = time.time()
env_40.train_with_buffer(
    agent_40, agent_40_2,
    n           = 600,
    train_every = 20,
    batch_size  = 256,
    epochs      = 3,
    ckpt_every  = 200,
    plot_every  = 50,
)
print(f'Train 40x40: {time.time()-t0:.0f}s')

In [ ]:
agent_40.save(os.path.join(CKPT_DIR, 'final_40.keras'))
env_40.play_n(agent_40, greedy_40, n=20)

---
## Xem kết quả

In [ ]:
env_40.play_one(agent_40, greedy_40, title='Agent vs Greedy')

In [ ]:
for f in sorted(os.listdir(CKPT_DIR)):
    size = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1024 / 1024
    print(f'  {f}  ({size:.1f} MB)')

---
## Load model đã lưu

In [ ]:
# agent = CaroAgent(board_size=40, base_filters=64, n_res=5, n_sim=50)
# agent.load('checkpoints/final_40.keras')

---
## Bảng tham số

| Tham số | Ở đâu | Ý nghĩa |
|---------|-------|----------|
| `base_filters`, `n_res` | `CaroAgent(...)` | Kích thước model (giống khi transfer) |
| `n_sim` | `CaroAgent(...)` | Số simulations MCTS |
| `n_parallel` | `CaroAgent(...)` | Số sim song song trên GPU |
| `buffer_size` | `CaroAgent(...)` | Buffer tối đa bao nhiêu positions |
| `n` | `env.train_with_buffer(...)` | Tổng số ván |
| `train_every` | `env.train_with_buffer(...)` | Chơi bao nhiêu ván rồi train |
| `batch_size` | `env.train_with_buffer(...)` | Batch size khi train |
| `epochs` | `env.train_with_buffer(...)` | Số epochs mỗi lần train |